# DiDAQt — Butterfly demonstration on FABRIC (BMv2)

This notebook builds the DiDAQt failover testbed on the **butterfly topology**
from the paper, using **real FABRIC nodes for every node of the butterfly** and
**BMv2** (`simple_switch`, via fablib's `Attestable_Switch`) as the software P4
data plane.

## Topology (defaults)

```
 16 senders            4 switch ranks x 8 = 32 BMv2 switches          8 receivers
 (8 pairs)         rank0      rank1      rank2      rank3
   snd1 \
   snd2  >--- sw0n0 --- sw1n0 --- sw2n0 --- sw3n0 --------------------- rcv0
   snd3 \        \  X   /   \  X  /   \  X  /
   snd4  >--- sw0n1 --- sw1n1 --- sw2n1 --- sw3n1 --------------------- rcv1
    ...          (butterfly cross-links: stage s flips bit K-1-s)       ...
   snd15\
   snd16 >--- sw0n7 --- sw1n7 --- sw2n7 --- sw3n7 --------------------- rcv7
```

* **Butterfly self-routing**: a frame's destination MAC is always a receiver's
  NIC MAC. At rank *s*, switch *i* forwards toward its *straight* next hop if
  bit `K-1-s` of the destination receiver index equals bit `K-1-s` of *i*, else
  toward its *cross* next hop (`i XOR 2^(K-1-s)`). After all `K` stages the frame
  arrives at rank-`K` switch = receiver index. This gives every sender a path to
  **any** receiver — the redundancy DiDAQt exploits for failover (future work).
* Each butterfly link is created with **`add_monitored_l2network`** (single-site
  **L2Bridge**), so a Crinkle **monitor** node sits transparently on every link
  and reports to the **analyzer** added with **`add_analyzer`**.
* Two **controller** nodes run in a main/follower HA pair (`controller_ha.py`):
  the follower takes over if the main stops heart-beating.

## This notebook's goal

Create the slice and **verify reachability**: every sender's workload reaches its
assigned receiver through the butterfly. The link is unidirectional L2 (no ping),
so senders emit the DiDAQt workload and each receiver confirms it received frames
from both of its senders. Introducing failures / measuring failover is later work.

> **Scale warning.** With the default full-spec dimensions this slice is large:
> 16 senders + 32 switches + 8 receivers + 2 controllers + 1 analyzer + ~72
> per-link monitors ≈ **130 nodes on a single site**. Shrink `K` /
> `SENDERS_PER_SW` in the config cell for a smoke test first.


## 0. Fablib setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()
fablib.show_config()

## 1. Configuration & butterfly math

All knobs live here so the notebook can be resumed and rescaled from one place.

In [ ]:
import os
import json
from concurrent import futures

# ---- Placement (single site: L2Bridge monitored links require one site) ----
SITE = "STAR"                     # pick a large site; see fablib.list_sites()

# ---- Butterfly dimensions ----
K              = 3                # butterfly dimension; WIDTH = 2**K
WIDTH          = 2 ** K          # switches per rank (8)
SWITCH_RANKS   = K + 1           # rank 0 .. K  (4 ranks, K=3 stages between them)
SENDERS_PER_SW = 2               # senders feeding each rank-0 switch (pairs of 2)

NUM_SENDERS   = WIDTH * SENDERS_PER_SW   # 16
NUM_RECEIVERS = WIDTH                     # 8
NUM_SWITCHES  = WIDTH * SWITCH_RANKS      # 32

# ---- Node sizing (kept small; BMv2 + sender/receiver are light) ----
SW_CORES,  SW_RAM,  SW_DISK  = 2, 4, 20   # per BMv2 switch
END_CORES, END_RAM, END_DISK = 2, 4, 10   # per sender / receiver / controller

# ---- Names / images / paths ----
SLICE_NAME = "didaqt-butterfly"
END_IMAGE  = "default_ubuntu_22"          # senders/receivers/controllers
NAME_PREFIX = "C"                          # Crinkle resource prefix (analyzer/monitors)

REPO       = os.path.abspath("..")         # repo root (this notebook lives in artifact/)
REMOTE_DIR = "/home/ubuntu/didaqt"         # upload target on end hosts
RUN_DIR    = "/tmp/didaqt_run"             # per-run logs on end hosts

P4_LOCAL   = os.path.join(REPO, "examples/p4/l2_forward_bmv2.p4")
P4_TABLE   = "MyIngress.l2_forward"        # table name as compiled (see show_tables)

# ---- DiDAQt runtime params ----
HB_PORT    = 9000                          # receiver -> controller heartbeat UDP port
HA_PORT    = 9100                          # controller <-> controller HA heartbeat port
SEND_RATE  = 500                           # frames/sec/sender (BMv2-friendly)
TEST_SECS  = 20                            # workload duration

# =====================================================================
# Butterfly helpers (pure functions of the config above)
# =====================================================================
def bit(x, b):
    return (x >> b) & 1

def cross_bit(stage):
    """The index bit flipped by the 'cross' edge at a given stage (0..K-1)."""
    return 1 << (K - 1 - stage)

def switch_ports(rank):
    """Ordered BMv2 port list for a switch at the given rank.
    BMv2 port index == position in this list (Attestable_Switch.start_switch)."""
    ins = [f"in_s{j}" for j in range(SENDERS_PER_SW)] if rank == 0 else ["in_str", "in_cross"]
    outs = ["out_str", "out_cross"] if rank < K else ["out_rx"]
    return ins + outs

def sw_name(rank, idx):
    return f"sw{rank}n{idx}"

def snd_name(sid):
    return f"snd{sid}"

def rcv_name(r):
    return f"rcv{r}"

# sender ids are 1..NUM_SENDERS; pair p owns ids [p*SPS+1 .. p*SPS+SPS],
# and every sender in pair p is initially routed to receiver p.
def pair_of_sender(sid):
    return (sid - 1) // SENDERS_PER_SW

def egress_index(rank, node_idx, recv_idx, ports):
    """Return the BMv2 egress port index for frames destined to receiver
    recv_idx at switch (rank, node_idx), or None to drop (default)."""
    if rank == K:
        return ports.index("out_rx") if recv_idx == node_idx else None
    b = K - 1 - rank
    straight = bit(node_idx, b) == bit(recv_idx, b)
    return ports.index("out_str" if straight else "out_cross")

# ---- Report ----
n_sender_links = NUM_SENDERS
n_stage_links  = K * WIDTH * 2
n_recv_links   = NUM_RECEIVERS
n_links = n_sender_links + n_stage_links + n_recv_links
n_nodes = NUM_SENDERS + NUM_SWITCHES + NUM_RECEIVERS + 2 + 1 + n_links  # +ctrlx2 +analyzer +monitors

print(f"Butterfly K={K}  WIDTH={WIDTH}  switch ranks={SWITCH_RANKS} (stages={K})")
print(f"  senders={NUM_SENDERS}  switches={NUM_SWITCHES}  receivers={NUM_RECEIVERS}")
print(f"  links (each a monitored L2Bridge): sender={n_sender_links} "
      f"stage={n_stage_links} receiver={n_recv_links}  total={n_links}")
print(f"  approx FABRIC nodes (incl. {n_links} monitors, 2 controllers, 1 analyzer): {n_nodes}")
print(f"  site={SITE}")

## 2. Build the Crinkle slice

Order matters: `add_analyzer` must be called **before** any `add_monitored_l2network`.
We create the analyzer, all endpoint nodes and BMv2 switches, the out-of-band
control L3 network, and finally wire every butterfly link as a monitored L2Bridge
(with the downstream interface marked as a **sink** so the Crinkle trailer is
stripped before it reaches the next hop / receiver).

In [ ]:
slice = fablib.new_crinkle_slice(name=SLICE_NAME, name_prefix=NAME_PREFIX)

# --- Analyzer first (required before monitored networks) ---
slice.add_analyzer(site=SITE)

# --- BMv2 switches: sw{rank}n{idx}, ports ordered per switch_ports() ---
switches = {}                      # (rank, idx) -> Attestable_Switch
for rank in range(SWITCH_RANKS):
    for idx in range(WIDTH):
        sw = slice.add_attestable_switch(
            name=sw_name(rank, idx), site=SITE, ports=switch_ports(rank),
            cores=SW_CORES, ram=SW_RAM, disk=SW_DISK,
        )
        switches[(rank, idx)] = sw

# --- Senders: one NIC_Basic 'd' (data) each ---
senders = {}                       # sid -> node
for sid in range(1, NUM_SENDERS + 1):
    n = slice.add_node(name=snd_name(sid), site=SITE,
                       cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
    n.add_component(model="NIC_Basic", name="d")   # data-plane iface (only NIC a sender needs)
    senders[sid] = n

# --- Receivers: 'd' (data) + 'c' (control) ---
receivers = {}                     # r -> node
for r in range(NUM_RECEIVERS):
    n = slice.add_node(name=rcv_name(r), site=SITE,
                       cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
    n.add_component(model="NIC_Basic", name="d")
    n.add_component(model="NIC_Basic", name="c")
    receivers[r] = n

# --- Controllers: main + follower, control iface only ---
ctl_main = slice.add_node(name="ctlmain", site=SITE,
                          cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
ctl_main.add_component(model="NIC_Basic", name="c")
ctl_follow = slice.add_node(name="ctlfollow", site=SITE,
                            cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
ctl_follow.add_component(model="NIC_Basic", name="c")

# --- Out-of-band control network (FABNetv4) for heartbeats ---
ctrl_ifaces = [receivers[r].get_component(name="c").get_interfaces()[0] for r in receivers]
ctrl_ifaces += [ctl_main.get_component(name="c").get_interfaces()[0],
                ctl_follow.get_component(name="c").get_interfaces()[0]]
ctrl_net = slice.add_l3network(name="ctlnet", interfaces=ctrl_ifaces, type="IPv4")

def data_iface(node):
    return node.get_component(name="d").get_interfaces()[0]

# --- Butterfly links (each a monitored L2Bridge; sink = downstream iface) ---
def link(name, up_iface, down_iface):
    up_iface.set_mode("manual")
    down_iface.set_mode("manual")
    slice.add_monitored_l2network(
        name=name, interfaces=[up_iface, down_iface], sinks=[down_iface],
        site=SITE,
    )

# 1) sender -> rank-0 switch
for p in range(WIDTH):
    ports = switch_ports(0)
    for j in range(SENDERS_PER_SW):
        sid = p * SENDERS_PER_SW + j + 1
        sw_if = switches[(0, p)].get_port_interface(ports[j])   # in_s{j}
        link(f"lsnd{sid}", data_iface(senders[sid]), sw_if)

# 2) inter-rank butterfly stages
for s in range(K):
    for i in range(WIDTH):
        up = switches[(s, i)]
        # straight edge: (s,i).out_str -> (s+1,i).in_str
        link(f"l{s}str{i}",
             up.get_port_interface("out_str"),
             switches[(s + 1, i)].get_port_interface("in_str"))
        # cross edge: (s,i).out_cross -> (s+1, i^bit).in_cross
        j = i ^ cross_bit(s)
        link(f"l{s}x{i}",
             up.get_port_interface("out_cross"),
             switches[(s + 1, j)].get_port_interface("in_cross"))

# 3) rank-K switch -> receiver
for r in range(WIDTH):
    link(f"lrcv{r}",
         switches[(K, r)].get_port_interface("out_rx"),
         data_iface(receivers[r]))

print(f"Built slice with {len(slice.get_nodes())} experiment nodes + "
      f"monitors/analyzer. Ready to submit.")

## 3. Submit

Crinkle's `submit()` places monitors on different workers from the nodes they watch, then runs post-boot config (this is the slow step for a large slice).

In [ ]:
slice.submit()
print("Slice active.")

## 4. Discover MACs / devices and compute forwarding rules

After boot we read each receiver's **data NIC MAC** (the destination every sender
targets) and each switch port's OS device name, then compute the per-switch
`l2_forward` table entries from the butterfly self-routing rule.

In [ ]:
# Refresh handles from the live slice.
switches = {(rank, idx): slice.get_attestable_switch(name=sw_name(rank, idx))
            for rank in range(SWITCH_RANKS) for idx in range(WIDTH)}
receivers = {r: slice.get_node(name=rcv_name(r)) for r in range(NUM_RECEIVERS)}
senders   = {sid: slice.get_node(name=snd_name(sid)) for sid in range(1, NUM_SENDERS + 1)}
ctl_main   = slice.get_node(name="ctlmain")
ctl_follow = slice.get_node(name="ctlfollow")

# Receiver data-plane MACs + devices.
rx_mac = {}      # r -> data NIC MAC (frames are addressed to this)
rx_dev = {}      # r -> data NIC OS device
for r in range(NUM_RECEIVERS):
    dif = receivers[r].get_component(name="d").get_interfaces()[0]
    rx_mac[r] = dif.get_mac()
    rx_dev[r] = dif.get_device_name()

# Sender data-plane devices + the receiver each sender targets.
snd_dev = {}     # sid -> data NIC OS device
snd_target = {}  # sid -> receiver index (== its pair index)
for sid in range(1, NUM_SENDERS + 1):
    snd_dev[sid] = senders[sid].get_component(name="d").get_interfaces()[0].get_device_name()
    snd_target[sid] = pair_of_sender(sid)

# Control-plane IPs.
ctl_main_ip   = ctl_main.get_interface(network_name="ctlnet").get_ip_addr()
ctl_follow_ip = ctl_follow.get_interface(network_name="ctlnet").get_ip_addr()

# Per-switch forwarding rules: list of {"dst_mac", "port"} keyed by (rank, idx).
sw_rules = {}
for rank in range(SWITCH_RANKS):
    for idx in range(WIDTH):
        ports = switch_ports(rank)
        rules = []
        for r in range(NUM_RECEIVERS):
            eidx = egress_index(rank, idx, r, ports)
            if eidx is not None:
                rules.append({"dst_mac": rx_mac[r], "port": eidx})
        sw_rules[(rank, idx)] = rules

print("Receiver data MACs:")
for r in range(NUM_RECEIVERS):
    print(f"  rcv{r}: {rx_mac[r]}  dev={rx_dev[r]}")
print(f"main controller ip:   {ctl_main_ip}")
print(f"follower controller ip: {ctl_follow_ip}")
print(f"example rules for sw0n0: {sw_rules[(0,0)]}")

## 5. Program the BMv2 switches

For each switch: start `simple_switch`, hot-load the compiled `l2_forward_bmv2.p4`,
then install its forwarding entries via `simple_switch_CLI` (`run_command`). All 32
switches are configured in parallel.

In [ ]:
def mac_to_hex(mac):
    return "0x" + mac.replace(":", "").lower()

def program_switch(key):
    rank, idx = key
    sw = switches[key]
    name = sw_name(rank, idx)
    # 1) launch the (empty) simple_switch bound to this switch's ports
    sw.start_switch()
    # 2) compile + hot-swap our L2 program
    if not sw.load_program(P4_LOCAL):
        return name, False, "load_program failed"
    # 3) install forwarding entries
    installed = 0
    for rule in sw_rules[key]:
        cmd = f"table_add {P4_TABLE} forward {mac_to_hex(rule['dst_mac'])} => {rule['port']}"
        if sw.run_command(cmd):
            installed += 1
    return name, True, f"{installed}/{len(sw_rules[key])} rules"

results = {}
with futures.ThreadPoolExecutor(max_workers=16) as pool:
    jobs = {pool.submit(program_switch, key): key for key in switches}
    for job in futures.as_completed(jobs):
        name, ok, msg = job.result()
        results[name] = (ok, msg)
        print(f"  {name}: {'OK' if ok else 'FAIL'} — {msg}")

# Sanity: dump one switch's tables so you can confirm the table name / entries.
print("\n--- sw0n0 tables ---")
switches[(0, 0)].run_command("show_tables")
switches[(0, 0)].run_command(f"table_dump {P4_TABLE}")

## 6. Build DiDAQt on the end hosts and generate the topology YAML

Upload the repo to the senders, receivers and controllers, `make examples`, and
generate the butterfly `topology.yaml` the controller reads. The controller runs
in **log-only** mode for this milestone (no switch control channel yet).

In [ ]:
end_hosts = ([senders[s] for s in senders] +
             [receivers[r] for r in receivers] +
             [ctl_main, ctl_follow])

# --- deps ---
install_cmd = ("sudo apt-get update -qq && "
               "sudo apt-get install -y -qq build-essential ethtool libyaml-dev")
jobs = [n.execute_thread(install_cmd) for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
print("deps installed")

# --- upload repo + build ---
jobs = [n.upload_directory_thread(REPO, REMOTE_DIR) for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
jobs = [n.execute_thread(f"cd {REMOTE_DIR} && make examples") for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
print("didaqt built on end hosts")

In [ ]:
# Generate a butterfly topology.yaml for the DiDAQt controller.
# Schema follows examples/topology.yaml: a sequence of node maps with
# connections (keyed by local port) and initial_connections ({sender,receiver}).
import io

def yaml_conn(local_port, other_node, other_port, flows=None, bw="100G"):
    s = f"    {local_port}:\n"
    s += f"      other_node: {other_node}\n"
    s += f"      other_port: {other_port}\n"
    s += f"      max_bandwidth: {bw}\n"
    if flows:
        s += "      initial_connections:\n"
        for i, (snd, rcv) in enumerate(flows, 1):
            s += f"        {i}: {{ sender: {snd}, receiver: {rcv} }}\n"
    return s

# Precompute, for every switch, the straight/cross next hop and the set of
# initial flows crossing each port (initial routing = each sender -> its receiver
# along the self-routed straight/cross path).
def route_path(recv_idx):
    """Return the ordered list of (rank, idx) switches a frame to recv_idx takes,
    entering at rank-0 switch = recv_idx's pair (its own initial rank-0 node)."""
    # A sender in pair p targets receiver p, entering rank-0 node p.
    node = recv_idx  # initial rank-0 node index for this receiver's senders
    path = [(0, node)]
    for s in range(K):
        b = K - 1 - s
        if bit(node, b) != bit(recv_idx, b):
            node ^= (1 << b)
        path.append((s + 1, node))
    return path

# flows_on[(rank,idx)][port_name] = list of (sender_name, receiver_name)
from collections import defaultdict
flows_on = defaultdict(lambda: defaultdict(list))
for sid in range(1, NUM_SENDERS + 1):
    r = snd_target[sid]
    path = route_path(r)
    for hop in range(len(path) - 1):
        rank, idx = path[hop]
        nxt_rank, nxt_idx = path[hop + 1]
        port = "out_str" if nxt_idx == idx else "out_cross"
        flows_on[(rank, idx)][port].append((snd_name(sid), rcv_name(r)))
    # last hop into the receiver
    lr, li = path[-1]
    flows_on[(lr, li)]["out_rx"].append((snd_name(sid), rcv_name(r)))

buf = io.StringIO()
# senders
for sid in range(1, NUM_SENDERS + 1):
    p = pair_of_sender(sid)
    ports = switch_ports(0)
    in_port = 1 + (sid - 1) % SENDERS_PER_SW      # switch-side port number (1-based)
    buf.write(f"- name: {snd_name(sid)}\n")
    buf.write("  type: sender\n")
    buf.write(f"  sender_id: {sid}\n")
    buf.write("  sender_id_bytes: 26\n")
    buf.write("  max_bandwidth: 10G\n")
    buf.write(f"  initial_receiver: {rcv_name(snd_target[sid])}\n")
    buf.write(f"  group_id: {p + 1}\n")
    buf.write("  connections:\n")
    buf.write(yaml_conn(1, sw_name(0, p), in_port,
                        flows=[(snd_name(sid), rcv_name(snd_target[sid]))]))
# switches
for rank in range(SWITCH_RANKS):
    for idx in range(WIDTH):
        ports = switch_ports(rank)
        buf.write(f"- name: {sw_name(rank, idx)}\n")
        buf.write("  type: switch\n")
        buf.write("  switch_type_group: bmv2\n")
        buf.write("  connections:\n")
        for pnum, pname in enumerate(ports, 1):
            # resolve peer for this port
            if pname.startswith("in_s") and pname[4:].isdigit():
                j = int(pname[4:]); sid = idx * SENDERS_PER_SW + j + 1
                other, oport = snd_name(sid), 1
            elif pname == "in_str":
                other, oport = sw_name(rank - 1, idx), 1 + switch_ports(rank - 1).index("out_str")
            elif pname == "in_cross":
                src = idx ^ cross_bit(rank - 1)
                other, oport = sw_name(rank - 1, src), 1 + switch_ports(rank - 1).index("out_cross")
            elif pname == "out_str":
                other, oport = sw_name(rank + 1, idx), 1 + switch_ports(rank + 1).index("in_str")
            elif pname == "out_cross":
                dst = idx ^ cross_bit(rank)
                other, oport = sw_name(rank + 1, dst), 1 + switch_ports(rank + 1).index("in_cross")
            elif pname == "out_rx":
                other, oport = rcv_name(idx), 1
            buf.write(yaml_conn(pnum, other, oport, flows=flows_on[(rank, idx)].get(pname)))
# receivers
for r in range(NUM_RECEIVERS):
    buf.write(f"- name: {rcv_name(r)}\n")
    buf.write("  type: receiver\n")
    buf.write(f"  receiver_id: {r}\n")
    buf.write("  connections:\n")
    inflows = flows_on[(K, r)].get("out_rx")
    buf.write(yaml_conn(1, sw_name(K, r), 1 + switch_ports(K).index("out_rx"), flows=inflows))

topo_yaml = buf.getvalue()
local_topo = os.path.join(RUN_DIR.replace("/tmp", "."), "topology.yaml")
os.makedirs(os.path.dirname(local_topo), exist_ok=True)
with open(local_topo, "w") as f:
    f.write(topo_yaml)
print(topo_yaml[:1200])
print("... (truncated)")

# upload to both controllers
for c in (ctl_main, ctl_follow):
    c.execute(f"mkdir -p {REMOTE_DIR}", quiet=True)
    c.upload_file(local_topo, f"{REMOTE_DIR}/topology.yaml")
print("topology.yaml uploaded to controllers")

## 7. Bring up data-plane interfaces

The sender/receiver data NICs are raw L2; bring them up (+promisc, VLAN offload off).

In [ ]:
def bringup(node, comp="d"):
    dev = node.get_component(name=comp).get_interfaces()[0].get_device_name()
    cmd = (f"sudo ip link set {dev} up; "
           f"sudo ip link set {dev} promisc on; "
           f"sudo ethtool -K {dev} txvlan off rxvlan off 2>/dev/null; true")
    return node.execute_thread(cmd)

jobs = [bringup(senders[s]) for s in senders] + [bringup(receivers[r]) for r in receivers]
for j in futures.as_completed(jobs):
    j.result()
print("data-plane interfaces up")

## 8. Run the reachability test

Launch, in order, the **controllers** (main + follower HA pair), the **receivers**
(each reporting heartbeats to the main controller), and the **senders** (each
emitting the rate-limited workload to its receiver's MAC for `TEST_SECS`). Output
is redirected to per-node log files under `RUN_DIR`.

In [ ]:
def sh(node, cmd):
    return node.execute_thread(f"mkdir -p {RUN_DIR}; {cmd}")

# 1) controllers (log-only DiDAQt controller wrapped by the HA script)
ctl_cmd = f"sudo ./build/controller topology.yaml {HB_PORT}"
sh(ctl_main,
   f"cd {REMOTE_DIR} && sudo nohup python3 ./artifact/controller_ha.py --role main "
   f"--peer {ctl_follow_ip} --hb-port {HA_PORT} -- {ctl_cmd} "
   f"> {RUN_DIR}/ctl_main.log 2>&1 &")
sh(ctl_follow,
   f"cd {REMOTE_DIR} && sudo nohup python3 ./artifact/controller_ha.py --role follower "
   f"--peer {ctl_main_ip} --hb-port {HA_PORT} -- {ctl_cmd} "
   f"> {RUN_DIR}/ctl_follow.log 2>&1 &")
import time
time.sleep(3)

# 2) receivers
for r in range(NUM_RECEIVERS):
    sh(receivers[r],
       f"cd {REMOTE_DIR} && sudo nohup ./build/receiver {rx_dev[r]} {r} "
       f"{ctl_main_ip} {HB_PORT} > {RUN_DIR}/rcv{r}.log 2>&1 &")
time.sleep(2)

# 3) senders (bounded workload)
for sid in range(1, NUM_SENDERS + 1):
    r = snd_target[sid]
    sh(senders[sid],
       f"cd {REMOTE_DIR} && sudo nohup timeout {TEST_SECS} ./build/sender "
       f"-r {SEND_RATE} {snd_dev[sid]} {rx_mac[r]} {sid} "
       f"> {RUN_DIR}/snd{sid}.log 2>&1 &")
print(f"workload running for ~{TEST_SECS}s ...")
time.sleep(TEST_SECS + 5)

# stop everything (leave BMv2 switches running)
for n in end_hosts:
    n.execute("sudo killall -q sender receiver controller heartbeat_monitor; "
              "sudo pkill -f controller_ha; true", quiet=True)
print("workload stopped")

## 9. Verify reachability

Pull each receiver's log, strip the ANSI TUI, and confirm each receiver saw **valid** frames from **both** of its assigned senders.

In [ ]:
import re
ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")

def strip_ansi(s):
    return ANSI.sub("", s)

def expected_senders(r):
    return [r * SENDERS_PER_SW + j + 1 for j in range(SENDERS_PER_SW)]

overall_ok = True
print(f"{'receiver':10} {'sender':8} {'valid':>10} {'status'}")
for r in range(NUM_RECEIVERS):
    out, _ = receivers[r].execute(f"cat {RUN_DIR}/rcv{r}.log", quiet=True)
    text = strip_ansi(out)
    # keep only the last rendered frame (after the final cursor-home)
    frame = text
    for sid in expected_senders(r):
        # find a table row that starts with this sender id
        valid, status = 0, "MISSING"
        for line in frame.splitlines():
            toks = line.split()
            if toks and toks[0] == str(sid):
                # row layout: <id> <valid> <invalid> <status>
                nums = [t for t in toks[1:] if t.isdigit()]
                if nums:
                    valid = int(nums[0])
                if any(s in line for s in ("OK", "FAULT", "MISSED")):
                    status = next(s for s in ("OK", "FAULT", "MISSED") if s in line)
        ok = valid > 0
        overall_ok &= ok
        flag = "" if ok else "   <-- NO FRAMES"
        print(f"rcv{r:<7} snd{sid:<5} {valid:>10} {status}{flag}")

print("\n=== REACHABILITY: " + ("PASS ===" if overall_ok else "FAIL ==="))
print("\n--- main controller (log-only) tail ---")
out, _ = ctl_main.execute(f"cat {RUN_DIR}/ctl_main.log", quiet=True)
print(strip_ansi(out)[-1500:])

## 10. Teardown

Stop the switches and (optionally) delete the slice.

In [ ]:
# Stop BMv2 on every switch.
for key, sw in switches.items():
    try:
        sw.stop_switch()
    except Exception as e:
        print(f"{sw_name(*key)}: {e}")

# Uncomment to delete the slice:
# slice.delete()
# print("slice deleted")